In [1]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


## Simple Multi-Agent Design Pattern

### Prerequisite:
Define/Create all the tools required by the system

### 1. Create multiple agents using create_agent calls, each with its own list of tools respectively...
Each subagent is created with `create_agent(...)`:

- `subagent_1` gets access only to `square_root`: `subagent_1 = create_agent(tools=[square_root])`
- `subagent_2` gets access only to `square`: : `subagent_2 = create_agent(tools=[square_root])`

### 2. Wrap each subagent as a tool (apply the @tool decorator)
To let another agent use these specialists, each subagent is wrapped inside a new tool (using @tool decorator):

- `call_subagent_1(x)` sends a message to `subagent_1`
- `call_subagent_2(x)` sends a message to `subagent_2`

This means the main agent does not directly perform the math. Instead, it delegates the work to the appropriate subagent.

### 3. Create a main coordinating agent
Finally, `main_agent` is created with:

- the same LLM
- the tools `call_subagent_1` and `call_subagent_2`
- a system prompt telling it when to use them

```mermaid
graph TD
    A["Main Agent<br/>(Coordinator)"] -->|delegates| B["call_agent_square_root<br/>(Tool)"]
    A -->|delegates| C["call_agent_square<br/>(Tool)"]
    
    B -->|invokes| D["Subagent 1<br/>(square_root specialist)"]
    C -->|invokes| E["Subagent 2<br/>(square specialist)"]
    
    D -->|uses| F["square_root tool"]
    E -->|uses| G["square tool"]
    
    F -->|returns| D
    G -->|returns| E
    
    D -->|returns result| B
    E -->|returns result| C
    
    B -->|returns to| A
    C -->|returns to| A
    
    A -->|final answer| H["User"]
    
    style A fill:#4A90E2
    style B fill:#7ED321
    style C fill:#7ED321
    style D fill:#F5A623
    style E fill:#F5A623
    style F fill:#BD10E0
    style G fill:#BD10E0
```

## Creating subagents

In [3]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [4]:
agent_square_root = create_agent( model=llm_noreason, 
    tools=[square_root]
)

agent_square = create_agent( model=llm_noreason,
    tools=[square]
)

In [5]:
@tool
def call_agent_square_root(x: float) -> float:
    """Call agent_square_root in order to calculate the square root of a number"""
    response = agent_square_root.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_agent_square(x: float) -> float:
    """Call agent_square in order to calculate the square of a number"""
    response = agent_square.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=llm,
    tools=[call_agent_square_root, call_agent_square],
    system_prompt="You are a helpful assistant who can call subagents to " \
                   "calculate the square root or square of a number.")

## Test

In [6]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})
pp(response['messages'][-1].content)

[
    {
        'summary': [
            {
                'text': 'The user asked for the square root of 456. I called the square root agent and got the result: approximately 21.354. I should provide this answer to the user.',
                'type': 'summary_text',
            },
        ],
        'type': 'reasoning',
    },
    {
        'text': 'The square root of 456 is approximately 21.354.',
        'type': 'text',
    },
]


In [7]:
question = "What is the square of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})
pp(response['messages'][-1].content)

[
    {
        'summary': [
            {
                'text': 'The user asked for the square of 456. I called the agent_square function with x=456, and it returned 207,936.0. I should present this result clearly to the user.',
                'type': 'summary_text',
            },
        ],
        'type': 'reasoning',
    },
    {'text': 'The square of 456 is 207,936.', 'type': 'text'},
]


In [8]:
response['messages'][1].tool_calls[0]

{'name': 'call_agent_square',
 'args': {'x': '456'},
 'id': 'call_6f4f26d1-4c62-48a8-8d67-cc37ccc56607',
 'type': 'tool_call'}